In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("../data/processed/cleaned_data.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (8523, 13)


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,Store_Age
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380,14
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,4
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700,14
3,FDX07,19.20,Regular,0.022911,Fruits and Vegetables,182.0950,OUT010,1998,Medium,Tier 3,Grocery Store,732.3800,15
4,NCD19,8.93,Low Fat,0.016164,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,26


In [2]:
df["Store_Age"] = 2024 - df["Outlet_Establishment_Year"]

In [3]:
df["Item_Visibility"] = df["Item_Visibility"].replace(0, df["Item_Visibility"].mean())

In [4]:
df["Store_Age"] = 2024 - df["Outlet_Establishment_Year"]

In [5]:
# Features (same as your app)
features = [
    "Item_MRP",
    "Item_Visibility",
    "Item_Weight",
    "Outlet_Establishment_Year",
    "Store_Age",
    "Item_Type",
    "Outlet_Type",
    "Outlet_Location_Type",
    "Outlet_Size",
    "Item_Fat_Content"
]

# Input (X)
X = pd.get_dummies(df[features])
# Target (y)
y = df["Item_Outlet_Sales"]
print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Shape: (8523, 33)
Target Shape: (8523,)


In [6]:
import numpy as np
y = np.log1p(y)

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (6818, 33)
Test Shape: (1705, 33)


In [8]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


In [9]:
y_pred = model.predict(X_test)

print(y_pred[:10])

[6.94511139 6.74198577 6.72539536 8.29066308 8.02966673 6.59069989
 8.62466891 7.36276018 7.02335849 7.75623064]


In [10]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R2 Score:", r2)

RMSE: 0.5322289042133552
R2 Score: 0.7305662264472743


In [11]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=15,
    min_samples_split=5,
    random_state=42
)

rf.fit(X_train, y_train)

print("Random Forest trained")

Random Forest trained


In [12]:
y_pred_rf = rf.predict(X_test)

from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("RMSE:", rmse_rf)
print("R2:", r2_rf)

RMSE: 0.5361016660950325
R2: 0.7266308926832521


In [13]:
from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

print("Gradient Boosting trained")

Gradient Boosting trained


In [14]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Generate predictions using the trained Gradient Boosting model
y_pred_gbr = model.predict(X_test)

# 2. Calculate metrics directly
rmse_gbr = np.sqrt(mean_squared_error(y_test, y_pred_gbr))
r2_gbr = r2_score(y_test, y_pred_gbr)

print("GBR RMSE:", rmse_gbr)
print("GBR R2:", r2_gbr)

GBR RMSE: 0.5338618672295363
GBR R2: 0.7289103586757091


In [15]:
# Convert log-transformed values back to original sales scale

y_test_original = np.expm1(y_test)
y_pred_original = np.expm1(y_pred_gbr)

# Calculate metrics on original sales scale

rmse_original = np.sqrt(
    mean_squared_error(y_test_original, y_pred_original)
)

r2_original = r2_score(
    y_test_original,
    y_pred_original
)

print("Original Scale RMSE:", rmse_original)
print("Original Scale R2:", r2_original)

Original Scale RMSE: 1095.4170325123414
Original Scale R2: 0.5585166702693659


## Model Comparison on Original Sales Scale

Since the target variable was log-transformed using `log1p`, predictions were converted back to the original sales scale using `expm1` before final evaluation.

In [16]:
# Compare all models on original sales scale

linear_pred_original = np.expm1(y_pred)
rf_pred_original = np.expm1(y_pred_rf)
gbr_pred_original = np.expm1(y_pred_gbr)

y_test_original = np.expm1(y_test)

models = {
    "Linear Regression": linear_pred_original,
    "Random Forest": rf_pred_original,
    "Gradient Boosting": gbr_pred_original
}

for name, predictions in models.items():
    rmse = np.sqrt(
        mean_squared_error(y_test_original, predictions)
    )
    
    r2 = r2_score(
        y_test_original, predictions
    )
    
    print(f"{name}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R2 Score: {r2:.4f}")
    print("-" * 30)

Linear Regression
RMSE: 1084.56
R2 Score: 0.5672
------------------------------
Random Forest
RMSE: 1090.79
R2 Score: 0.5622
------------------------------
Gradient Boosting
RMSE: 1095.42
R2 Score: 0.5585
------------------------------


### Mean Absolute Error Evaluation

MAE was calculated to compare the average absolute prediction error of each model on the original sales scale.

In [17]:
from sklearn.metrics import mean_absolute_error

for name, predictions in models.items():
    mae = mean_absolute_error(
        y_test_original,
        predictions
    )

    print(f"{name}")
    print(f"MAE: {mae:.2f}")
    print("-" * 30)

Linear Regression
MAE: 744.81
------------------------------
Random Forest
MAE: 745.79
------------------------------
Gradient Boosting
MAE: 746.82
------------------------------


## Final Model Selection

Three regression models were evaluated on the original sales scale using RMSE, R² Score, and MAE.

Linear Regression achieved the lowest RMSE and highest R² score among the evaluated models. Therefore, Linear Regression was selected as the final model.

**Selected Model: Linear Regression**

In [18]:
from sklearn.linear_model import LinearRegression
import joblib

# Train final selected model
final_model = LinearRegression()
final_model.fit(X_train, y_train)

# Save final model
joblib.dump(final_model, "../models/final_model.pkl")

print("Final Linear Regression model saved successfully")

Final Linear Regression model saved successfully


In [19]:
import sklearn
print(sklearn.__version__)

1.8.0
